# verify02: モデルA（痛み/しびれ/振る舞い） vs 遷移BERT(B, D) を「ノード別精度」と「トリアージ」で対決

全モデルが `dataset/input_pairs.csv` の**同じ採用ペア**を使い、前処理・fold・ハイパラ共通。

| モデル | 学習のしかた | 入力テキスト |
|---|---|---|
| **モデルA（痛み/しびれ/振る舞い）** | 各ノード**専用**の単体分類器。`HeadacheBERT_painful_Finetuning.ipynb` の手法を痛み/しびれ/振る舞いに適用（162例ずつ・計3本） | 採用ペア（質問+回答まるごと・ノード質問なし） |
| **B. 遷移(質問+ペア)** | 3ノード全部を**1本で共有**学習（486例） | 質問 + 採用ペア |
| **D. 遷移(質問+回答だけ)** | 3ノード全部を**1本で共有**学習（486例） | 質問 + そのcos回答（回答列だけ） |

**出力は2つ**：
1. **ノード別 accuracy / macro-F1**（痛み/しびれ/振る舞い）… モデルA vs B vs D
2. **トリアージ accuracy**（R3/R2/Y2）… 各モデルのノード予測で**決定木を辿った**最終トリアージ
   - モデルA は 痛み/しびれ/振る舞い の各単体予測を合わせて1つの「系」として辿る。

> 採用ペア列：`_1`=痛み / `_2`=しびれ / `_3`=振る舞い。ノード出力は 0=はい/1=いいえ/2=不明。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', 'input_pairs.csv')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.46.3', 'sentencepiece', 'fugashi',
                    'ipadic', 'unidic-lite', 'protobuf', 'accelerate', 'pyyaml'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', 'input_pairs.csv')
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

## 1.1 実験設定（A/B/D 共通）

In [ ]:
MODEL_NAME = 'cl-tohoku/bert-base-japanese-v3'
MAX_LEN = 256        # CPUで重ければ 64
EPOCHS = 3           # CPUで重ければ 1
LR = 2e-5
BATCH = 8
N_FOLDS = 5
USE_STOPWORDS = False
SEED = 42

import random, numpy as np


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'MAX_LEN={MAX_LEN} EPOCHS={EPOCHS} LR={LR} BATCH={BATCH} N_FOLDS={N_FOLDS}')

# 2. データ・前処理・決定木（branch_table）

- `make_patient_table`：採用ペア行の `text_col`（`ペア`=質問+回答 / `回答`=回答だけ）を連結。
- `branch_table`：protocol.yaml から頭痛の3分岐とトリアージ行き先を読む（トリアージ判定に使う）。

In [ ]:
import pandas as pd
import yaml

df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())
assert '回答' in df.columns and 'ペア' in df.columns

# --- protocol.yaml → 質問文 & 決定木 ---
_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
fallback_triage = _h['fallback']['if_all_symptom_questions_negative']


def _is_branch(n):
    return 'choices' in n and not n.get('metadata_only', False)


def _parse_choice(c):
    if c.get('triage'):
        return {'action': 'terminal', 'triage': c['triage']}
    if c.get('next'):
        return {'action': 'next', 'next_id': c['next']}
    return {'action': 'fallback', 'triage': fallback_triage}


branch_table = [{'id': n['id'], 'question': n['question'],
                 'choices': [_parse_choice(c) for c in n['choices']]}
                for n in _h['nodes'] if _is_branch(n)]
branch_ids = {b['id'] for b in branch_table}

# ノード(キー) ↔ 採用ペア列 / ラベル列 / branch_id / 質問
NODES = [
    {'key': '痛み',   'adopt': '採用ペア_ひし形_全通り_頭痛_1', 'label': '痛み',
     'bid': 'headache_sudden_severe'},
    {'key': 'しびれ', 'adopt': '採用ペア_ひし形_全通り_頭痛_2', 'label': 'しびれ',
     'bid': 'headache_numbness_paralysis'},
    {'key': '振る舞い', 'adopt': '採用ペア_ひし形_全通り_頭痛_3', 'label': '振る舞い',
     'bid': 'headache_abnormal_behavior'},
]
_qmap = {b['id']: b['question'] for b in branch_table}
for n in NODES:
    n['question'] = _qmap[n['bid']]

triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}
print('branch_table:', [b['id'] for b in branch_table], '| fallback:', fallback_triage)

# --- ストップワード除去（painful 流用）---
import fugashi
_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set(['の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
                            'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'])


def remove_stopwords(text):
    return ''.join(w.surface for w in _tagger(text) if w.surface not in STOPWORD_EXTRA_WORDS)


def make_patient_table(adopt_col, label_col, use_question=False, question='', text_col='ペア'):
    rows = []
    for pid, g in df.groupby('id'):
        adopted = g[g[adopt_col] == True]
        parts = adopted[text_col].astype(str).tolist()
        text = ' '.join(parts) if parts else '(発話なし)'
        if USE_STOPWORDS:
            text = remove_stopwords(text)
        if use_question:
            text = question + ' ' + text
        rows.append({'id': pid, 'text': text, 'label': int(g[label_col].iloc[0])})
    return pd.DataFrame(rows).set_index('id')

# 3. ★BERTの中身★（学習と予測。A/B/D 共通で使う）

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class PainTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels = list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


_tok_cache = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


def train_model(texts, labels):
    set_seed(SEED)                              # 1) 乱数固定
    tokenizer = get_tokenizer(MODEL_NAME)
    model = build_model(MODEL_NAME)             # 2) まっさらな3クラスBERT
    loader = DataLoader(PainTextDataset(texts, labels, tokenizer, MAX_LEN),
                        batch_size=BATCH, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    for epoch in range(EPOCHS):                 # 3) 予測→loss→逆伝播→更新
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    return model, tokenizer


@torch.no_grad()
def predict_labels(model, tokenizer, texts):
    model.eval()
    preds = []
    ds = PainTextDataset(texts, [0] * len(texts), tokenizer, MAX_LEN)
    for batch in DataLoader(ds, batch_size=BATCH, shuffle=False):
        batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        preds += model(**batch).logits.argmax(dim=-1).cpu().tolist()
    return preds


print('train_model / predict_labels 定義（A/B/D 共通コア）')

# 4. fold分割（患者単位5-fold・全モデル共通）

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

patients = np.array(sorted(df['id'].unique()))
triage = np.array([int(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage)]
print(f'{N_FOLDS}-fold。test患者数:', [len(te) for _, te in FOLDS])

# 各患者の正解（ノードラベル・トリアージ）
true_node = {n['key']: {p: int(df[df['id'] == p][n['label']].iloc[0]) for p in patients} for n in NODES}
true_triage = {p: triage_decode[int(df[df['id'] == p]['トリアージ'].iloc[0])] for p in patients}

# 5. 予測を集める

- **モデルA（痛み/しびれ/振る舞い）**：`HeadacheBERT_painful_Finetuning.ipynb` と同じ手法
  （単体分類器・採用ペア入力・質問なし・`PainTextDataset`/`build_model`/前処理を流用、ストップワード除去は `USE_STOPWORDS`）
  を各ノードに適用し、fold毎に学習・予測。
- **B / D**：3ノードを1本で共有学習し、各ノードを予測。

返すのはどれも `{ノード: {患者: 予測ラベル}}` の形（あとでノード別精度にもトリアージにも使える）。

In [ ]:
def collect_single(node):
    """モデルA: 1ノード専用の painful分類器（HeadacheBERT_painful_Finetuning と同手法）。
    採用ペア(ペア列)・質問なし。"""
    tab = make_patient_table(node['adopt'], node['label'], use_question=False, text_col='ペア')
    pred = {}
    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        model, tok = train_model(tab.loc[tr_ids, 'text'].tolist(), tab.loc[tr_ids, 'label'].tolist())
        yp = predict_labels(model, tok, tab.loc[te_ids, 'text'].tolist())
        for pid, p in zip(te_ids, yp):
            pred[pid] = p
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pred


def collect_shared(text_col, tag):
    """B/D: 3ノードを1本で共有学習（質問あり）。text_col で B(ペア)/D(回答) を切替。"""
    tabs = {n['key']: make_patient_table(n['adopt'], n['label'], True, n['question'], text_col) for n in NODES}
    pred = {n['key']: {} for n in NODES}
    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        train_texts, train_labels = [], []
        for n in NODES:
            t = tabs[n['key']]
            train_texts += t.loc[tr_ids, 'text'].tolist()
            train_labels += t.loc[tr_ids, 'label'].tolist()
        model, tok = train_model(train_texts, train_labels)
        for n in NODES:
            t = tabs[n['key']]
            yp = predict_labels(model, tok, t.loc[te_ids, 'text'].tolist())
            for pid, p in zip(te_ids, yp):
                pred[n['key']][pid] = p
        print(f'  [{tag}] fold{i} done')
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pred


print('モデルA（痛み/しびれ/振る舞い）を学習中 ...')
predA = {n['key']: collect_single(n) for n in NODES}
print('B（質問+ペア）を学習中 ...')
predB = collect_shared('ペア', 'B')
print('D（質問+回答だけ）を学習中 ...')
predD = collect_shared('回答', 'D')
print('全モデル予測 完了')

# 6. 結果①：ノード別 accuracy / macro-F1（A-i vs B vs D）

In [ ]:
def node_scores(pred_for_node, key):
    yt = [true_node[key][p] for p in patients]
    yp = [pred_for_node[p] for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


rows = []
for n in NODES:
    k = n['key']
    aA, fA = node_scores(predA[k], k)
    aB, fB = node_scores(predB[k], k)
    aD, fD = node_scores(predD[k], k)
    rows.append({'ノード': k,
                 'モデルA acc': f'{aA:.3f}', 'B遷移 acc': f'{aB:.3f}', 'D遷移 acc': f'{aD:.3f}',
                 'モデルA F1': f'{fA:.3f}', 'B遷移 F1': f'{fB:.3f}', 'D遷移 F1': f'{fD:.3f}'})
node_table = pd.DataFrame(rows)
print('===== ノード別 accuracy / macro-F1 =====')
display(node_table)
node_table.to_csv(os.path.join(OUT_DIR, 'verify02_node.csv'), index=False, encoding='utf-8-sig')

# 7. 結果②：トリアージ accuracy（ノード予測で決定木を辿る）

各患者について、モデルのノード予測（はい/いいえ/不明）で頭痛の決定木を辿り、最終トリアージ
（R3/R2/Y2）を決める。モデルA は 痛み/しびれ/振る舞い の各単体予測を合わせて1つの系として辿る。

In [ ]:
def predict_triage(pred_idx_by_bid):
    """ノードごとの予測choice(0/1/2)で決定木を辿り、最終トリアージを返す。"""
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        ch = b['choices'][pred_idx_by_bid[b['id']]]
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


def triage_scores(pred_by_key):
    yt, yp = [], []
    for p in patients:
        by_bid = {n['bid']: pred_by_key[n['key']][p] for n in NODES}
        yp.append(predict_triage(by_bid))
        yt.append(true_triage[p])
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


# サニティチェック：正解ノードラベルで辿ると真のトリアージに一致するはず
_gold = {n['key']: true_node[n['key']] for n in NODES}
_ga, _ = triage_scores(_gold)
print(f'[sanity] 正解ラベルで辿ったトリアージ一致率 = {_ga:.3f}（1.000 ならツリー実装OK）')

tri_rows = []
for name, pred in [('モデルA(痛み/しびれ/振る舞い)', predA), ('B 遷移(質問+ペア)', predB), ('D 遷移(質問+回答)', predD)]:
    a, f = triage_scores(pred)
    tri_rows.append({'モデル': name, 'トリアージ acc': f'{a:.3f}', 'トリアージ macro-F1': f'{f:.3f}'})
tri_table = pd.DataFrame(tri_rows)
print('\n===== トリアージ（最終R3/R2/Y2）=====')
display(tri_table)
tri_table.to_csv(os.path.join(OUT_DIR, 'verify02_triage.csv'), index=False, encoding='utf-8-sig')
print('saved: verify02_node.csv, verify02_triage.csv')

# 8. まとめ（読み方）

- **ノード別精度**：モデルA(各ノード162例・painful手法) vs B/D(共有486例)。データ量・入力形式の効きを見る。
- **トリアージ**：ノード予測で決定木を辿った最終R3/R2/Y2。3ノードを通した実運用に近い指標。
- **B vs D**：入力が「質問+ペア」か「質問+回答だけ」か。
- 採用ペア・fold・ハイパラは全モデル共通なので、差は学習のしかた／入力の作り方だけから来る。